<a href="https://colab.research.google.com/github/Amitabh-Phule/Deep-Learning/blob/main/StudyGenie_RAG_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧞‍♂️ StudyGenie – AI Powered PDF Study Assistant (LangChain Edition)
---
## 📌 Concept
StudyGenie is a smart learning assistant powered by a **LangChain-based RAG pipeline**.

It helps students interact with PDF documents efficiently using AI.

---

## ✨ Features (3 Wishes System)
1. ❓ Ask Questions (Context-based Q&A)
2. 📄 Generate Summary
3. 📘 Create Study Plan

### 📦 Step 1: Install Required Libraries

In [1]:
!pip install -q \
numpy \
faiss-cpu \
PyPDF2 \
sentence-transformers \
groq \
langchain \
langchain-community \
langchain-core \
langchain-huggingface

### 🔑 Step 2: Setup API and Import Libraries

In [2]:
import os
from google.colab import userdata

# Groq API Key
os.environ["GROQ_API_KEY"] = userdata.get("StudyGenie")

from groq import Groq
client = Groq()

import PyPDF2

from langchain_community.vectorstores import FAISS as LC_FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
from langchain.memory import ConversationBufferMemory

from google.colab import files

memory = ConversationBufferMemory(return_messages=True)

print("✅ Setup complete")

✅ Setup complete


### 🧠 Step 3: Load Embedding Model

In [3]:
print("⚙️ Loading embedding model...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding model ready")

⚙️ Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Embedding model ready


### 📂 Step 4: Upload PDF and Build Vector Store

In [5]:
def load_pdf_and_build_index():
    global vectorstore

    print("📂 Upload your PDF")
    uploaded = files.upload()

    pdf_path = list(uploaded.keys())[0]

    print("⚙️ Processing...")

    reader = PyPDF2.PdfReader(pdf_path)
    text = ""

    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted

    docs = [Document(page_content=text)]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    split_docs = splitter.split_documents(docs)

    global embeddings
    vectorstore = LC_FAISS.from_documents(split_docs, embeddings)
print("✅ PDF processed & ready")

✅ PDF processed & ready


### 🔍 Step 5: Retrieval Function

In [6]:
def retrieve_docs(query):
    docs = vectorstore.similarity_search(query, k=2)
    return " ".join([doc.page_content[:300] for doc in docs])

### 💬 Step 6: Question Answering System

In [7]:
MODEL_NAME = "llama-3.1-8b-instant"

def ask_question():
    query = input("Ask your question: ")

    context = retrieve_docs(query)
    history = memory.load_memory_variables({})["history"]

    prompt = f"""
    You are a helpful AI tutor.

    Use ONLY the provided context.
    If answer is not in context, say "Not found in document".

    Chat History:
    {history}

    Context:
    {context}

    Question:
    {query}
    """

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}]
    )

    answer = response.choices[0].message.content

    print("\n🤖 Answer:")
    print(answer)

    memory.save_context(
        {"input": query},
        {"output": answer}
    )

### 📄 Step 7: Generate Summary

In [8]:
def summarize_pdf():
    docs = vectorstore.similarity_search("summary of document", k=3)
    context = " ".join([doc.page_content[:500] for doc in docs])

    prompt = f"""
    Summarize the following document clearly:

    {context}
    """

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}]
    )

    print("\n📄 Summary:")
    print(response.choices[0].message.content)

### 📘 Step 8: Generate Study Plan

In [9]:
def study_plan():
    docs = vectorstore.similarity_search("important topics", k=3)
    context = " ".join([doc.page_content[:500] for doc in docs])

    prompt = f"""
    Create a SIMPLE study plan based on this content.

    Rules:
    - Do NOT divide into weeks
    - Only topic-wise list
    - Keep it short

    Format:
    1. Topic
    2. Topic
    """

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}]
    )

    print("\n📘 Study Plan:")
    print(response.choices[0].message.content)

### 🎮 Step 9: Run the Application

In [14]:
def run():
    print("🧞 StudyGenie Started")

    while True:

        if 'vectorstore' not in globals():
            print("\n1. Upload PDF\n0. Exit")
            ch = input("Choice: ")

            if ch == "1":
                load_pdf_and_build_index()
            else:
                print("🧞 Ended")
                break

        else:
            print("\n1. Q&A\n2. Summary\n3. Plan\n4. Exit")
            ch = input("Choice: ")

            if ch == "1":
                ask_question()
            elif ch == "2":
                summarize_pdf()
            elif ch == "3":
                study_plan()
            else:
                print("🧞 Ended")
                break
run()

🧞 StudyGenie Started

1. Upload PDF
0. Exit
Choice: 1
📂 Upload your PDF


Saving Unit-V.pdf to Unit-V (5).pdf
⚙️ Processing...

1. Q&A
2. Summary
3. Plan
4. Exit
Choice: 1
Ask your question: what is fine tuning

🤖 Answer:
Fine-tuning is a technique in machine learning where a pre-trained model is further trained on a new, usually smaller and task-specific dataset to improve its performance for a particular task.

1. Q&A
2. Summary
3. Plan
4. Exit
Choice: 2

📄 Summary:
**Large-Scale Dataset Creation and Generation**

This document describes the process of creating and generating large-scale datasets using Artificial Intelligence (AI) models. The process involves the following steps:

1. **Collect outputs**: Collecting data or text samples from various sources.
2. **Filter/Clean results**: Cleaning and filtering the collected data to ensure accuracy and quality.
3. **(Optional) Human review**: Reviewing the generated data by humans to check for accuracy and quality.

**Advantages of AI-based Dataset Creation**

1. **Fast**: Generating millions of samples quick